In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import matplotlib.pyplot as plt
import os

DATA_PATH = ".\data\opsd_building.csv"

In [4]:
def load_single_file(path):
    """Load and parse a single data file"""
    df = pd.read_csv(
        path,
        usecols=[1,2,3,4],  # Read all 4 columns directly
        header=0,           # First row is header, skip automatically
        names=['load1', 'load2', 'load3', 'price'],
        parse_dates=False   # Disable auto-parsing when no datetime column exists
    )
    # Generate time series (assuming data is ordered chronologically)
    start_time = pd.to_datetime('2020-01-01 00:00:00')  # Modify according to actual requirements
    df['timestamp'] = start_time + pd.to_timedelta(df.index*15, 'm')
    df.set_index('timestamp', inplace=True)
    # Validate data integrity
    assert len(df)%96 == 0, "Incomplete daily data"
    print(f"Successfully loaded {len(df)/96:.1f} days of data")
    return df

raw_df = load_single_file(DATA_PATH)
display(raw_df.head())

Successfully loaded 661.0 days of data


,load1,load2,load3,price
timestamp,,,,
2020-01-01 00:00:00,0.079,0.187,0.039,29.93
2020-01-01 00:15:00,0.078,0.000,0.031,29.93
2020-01-01 00:30:00,0.080,0.000,0.075,29.62
2020-01-01 00:45:00,0.110,0.000,0.176,29.62
2020-01-01 01:00:00,0.107,0.135,0.541,29.62


In [5]:
def generate_daily_samples(df, lookback_days=3):
    """
    Generate daily samples (with historical window)
    Parameters:
    lookback_days : Number of historical days (0 means only current day data)
    Returns:
    Sample array with shape (num_days, 96, features)
    """
    day_length = 96
    total_days = len(df) // day_length
    print(f"Total days in data: {total_days} days")
    samples = []
    for day in range(lookback_days, total_days-1):  # -1 keeps last day as target
        # Get historical window data
        start_idx = (day - lookback_days) * day_length
        end_idx = (day + 1) * day_length  # +1 includes current day
        # Extract feature matrix [timesteps × features]
        sample_data = df.iloc[start_idx:end_idx][['load1', 'load2', 'load3','price']]
        print(sample_data)
        # Validate sample integrity
        assert len(sample_data) == (lookback_days+1)*day_length, \
            "Incorrect sample length for day {}".format(day)
        samples.append(sample_data.values.reshape(-1, day_length, 4))
    
    return np.vstack(samples)
# Generate samples with 3-day historical window
daily_samples = generate_daily_samples(raw_df, lookback_days=3)
print("Sample shape:", daily_samples.shape)  # (num_samples, history_days+1, 96_timesteps, 9_features)


Total days in data: 661 days
                     load1  load2  load3  price
timestamp                                      
2020-01-01 00:00:00  0.079  0.187  0.039  29.93
2020-01-01 00:15:00  0.078  0.000  0.031  29.93
2020-01-01 00:30:00  0.080  0.000  0.075  29.62
2020-01-01 00:45:00  0.110  0.000  0.176  29.62
2020-01-01 01:00:00  0.107  0.135  0.541  29.62
...                    ...    ...    ...    ...
2020-01-04 22:45:00  0.103  0.000  0.022  21.09
2020-01-04 23:00:00  0.098  0.000  0.036  21.09
2020-01-04 23:15:00  0.070  0.000  0.041  21.09
2020-01-04 23:30:00  0.130  0.000  0.278  19.90
2020-01-04 23:45:00  0.099  0.000  0.378  19.90

[384 rows x 4 columns]
                     load1  load2  load3  price
timestamp                                      
2020-01-02 00:00:00  0.072   0.03  0.050  27.93
2020-01-02 00:15:00  0.090   0.01  0.059  27.93
2020-01-02 00:30:00  0.088   0.01  0.071  28.42
2020-01-02 00:45:00  0.112   0.00  0.042  28.42
2020-01-02 01:00:00  0.105   0.00  